# Genie BEFORE: Confirming the Structural Gap

**Run on serverless compute.**

Genie is a high-quality translator of analyst questions into SQL. Against the base Silver tables, it handles account balances, transfer volumes, merchant categories, regional activity, and top-account lists without difficulty. That is the baseline this notebook establishes first: Genie doing its designed job well on the catalog the customer already runs.

The structural questions in this notebook belong to a different category. Finding transfer-network hubs, finding accounts that form communities based on how densely they transact with each other, finding accounts that share merchant relationships without transacting directly: those answers live in network topology. No row-level aggregation can produce them. When Genie attempts these questions against Silver tables, the misses are not failures of the tool. They are evidence that the information does not yet exist in the data.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"
VOLUME   = "graph-enriched-volume"

SPACE_ID = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_before")

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import (
    genie_caller,
    load_ground_truth,
    check_risk_score_precision,
)

w = WorkspaceClient()
print("Connected to:", w.config.host)
ask_genie = genie_caller(w, SPACE_ID)

In [ ]:
_GT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"
gt = load_ground_truth(_GT_PATH)

print(
    f"Ground truth: {gt['summary']['total_rings']} rings, "
    f"{gt['summary']['total_fraud_accounts']:,} fraud accounts, "
    f"{gt['summary']['total_whale_accounts']:,} whales"
)

## Warm-Up: Tabular Baseline

The first question has a clear tabular answer: rank accounts by total spend.
Genie resolves this with a straightforward aggregation. The result confirms
Genie is working and establishes a baseline before the structural questions run.

In [ ]:
WARMUP_QUESTION = "What are the top 10 accounts by total amount spent across all merchants?"

print("[W] warm_up — asking...")
response_warmup = ask_genie(WARMUP_QUESTION)

if response_warmup["df"] is not None:
    print(f"    Rows: {len(response_warmup['df'])}")
    print(f"    SQL:  {response_warmup['sql']}")
    display(response_warmup["df"])
else:
    print(f"    Status: {response_warmup['status']}")
    print(f"    Text:   {response_warmup['text']}")

print()
print("[W] warm_up — TABULAR BASELINE")
print("    Confirms Genie answers basic tabular aggregations correctly.")

## Analytics Challenge

A harder tabular question before the structural questions: identify accounts with both elevated total spend and above-average night activity. Night transactions are hours 0 through 5, an established fraud signal. Genie should resolve this with a join and a conditional aggregate over the base tables.

**Expected outcome:** a clean top-15 list ordered by total spend with night ratio and balance columns. Genie handles this correctly because all three dimensions live in the base tables.

In [ ]:
ANALYTICS_QUESTION = (
    "Which accounts have both above-average total spend and a night transaction ratio "
    "above 20%? Show the top 15 by total spend with their night ratio and account balance."
)

print("[A] analytics_challenge — asking...")
response_analytics = ask_genie(ANALYTICS_QUESTION)

if response_analytics["df"] is not None:
    print(f"    Rows: {len(response_analytics['df'])}")
    print(f"    SQL:  {response_analytics['sql']}")
    display(response_analytics["df"])
else:
    print(f"    Status: {response_analytics['status']}")
    print(f"    Text:   {response_analytics['text']}")

print()
print("[A] analytics_challenge — TABULAR QUERY RESOLVED")
print("    Genie joined transactions and accounts and applied a conditional aggregate.")
print("    All dimensions exist in the base tables.")

## Structural Question

The next question targets structure that lives in the transfer network topology: hub positions. This property is not encoded in any column the base tables carry. The result is labeled `STRUCTURAL GAP CONFIRMED` or `UNEXPECTED SIGNAL FOUND` based on how well the response aligns with the known fraud-ring ground truth.

### Question 1: Hub Detection

An analyst investigating fraud would ask which accounts sit at the center of a
money-movement network. On the base catalog, Genie has no `risk_score` column to
rank by. Only raw transaction counts and amounts are available. The result surfaces
high-volume legitimate accounts alongside ring members with no signal to
separate them.

**Expected outcome:** low precision at top-20; the structural gap is confirmed.

In [ ]:
HUB_QUESTION = (
    "Are there accounts that seem to be the hub of a money movement "
    "network that are potentially fraudulent?"
)

print("[1] hub_detection — asking...")
response_hub = ask_genie(HUB_QUESTION)

if response_hub["df"] is not None:
    print(f"    Rows: {len(response_hub['df'])}")
    print(f"    SQL:  {response_hub['sql']}")
    display(response_hub["df"])

    result_hub = check_risk_score_precision(response_hub["df"], gt, topn=20)
    verdict_hub = "UNEXPECTED SIGNAL FOUND" if result_hub["passed"] else "STRUCTURAL GAP CONFIRMED"

    print()
    print(f"[1] hub_detection — {verdict_hub}")
    print(f"    Metric:  precision={result_hub['precision']:.2f}  (after-GDS criterion: > 0.70)")
    print(f"    Finding: {result_hub['true_positives']}/{result_hub['topn']} top-{result_hub['topn']} accounts are known fraud ring members")
else:
    verdict_hub = "STRUCTURAL GAP CONFIRMED"
    print(f"    Status: {response_hub['status']}")
    print(f"    Text:   {response_hub['text']}")
    print()
    print(f"[1] hub_detection — {verdict_hub}")
    print("    Genie returned no tabular result; the base catalog has no score column to rank by.")

In [ ]:
print("=" * 78)
print("Genie BEFORE — Summary")
print("=" * 78)
print()
print("Purpose: confirm structural gap on base tables (misses are expected evidence)")
print()
print("  [W] warm_up              TABULAR BASELINE")
print("  [A] analytics_challenge  TABULAR QUERY RESOLVED")
print(f"  [1] hub_detection        {verdict_hub}")
print()
print("-" * 78)
print("The base catalog cannot resolve structural questions from row-level SQL.")
print("Enrichment unlocks portfolio, cohort, community, operational, and merchant")
print("questions in the AFTER run.")

## Next Steps

Proceed to **`02_neo4j_ingest`** to load the five base tables into Neo4j as a
property graph. After the full pipeline (`02_neo4j_ingest` → `03_gds_enrichment`
→ `04_pull_gold_tables`), open `05_genie_after` to ask analyst questions against
the enriched catalog.